In [ ]:
__trill_node__ = {
    "id": "09c6e03f-117d-45c5-af30-8f71bc3e58b6",
    "type": "COMPUTATION_ANALYSIS",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():



    # computation analysis - clear

    import sys
    import os
    import re
    import torch

    _ROOT = os.getcwd()
    _CANDIDATES = [
        os.environ.get("CITYSURFACES_DIR"),
        os.path.join(_ROOT, "city-surfaces"),
        os.path.join(_ROOT, "CitySurfaces"),
    ]
    CITYSURFACES_DIR = None
    for _p in _CANDIDATES:
        if _p and os.path.isfile(os.path.join(_p, "config.py")):
            CITYSURFACES_DIR = os.path.abspath(_p)
            break
    if CITYSURFACES_DIR is None:
        raise FileNotFoundError(
            "CitySurfaces repo not found (need config.py). Clone "
            "https://github.com/VIDA-NYU/city-surfaces into ./city-surfaces/ under "
            f"your Curio launch directory, or set CITYSURFACES_DIR. cwd={_ROOT!r}"
        )
    sys.path.insert(0, CITYSURFACES_DIR)

    # CitySurfaces calls logx.msg() during model init; runx requires initialize() first (normally done in val.py main).
    import tempfile
    from runx.logx import logx
    _log_dir = os.path.join(tempfile.gettempdir(), "curio_citysurfaces_runx")
    logx.initialize(logdir=_log_dir, tensorboard=False, hparams={}, global_rank=0)

    WEIGHTS_DIR = './data/dataset/CitySurfaces_weights'
    WEIGHTS_FILE = os.path.join(WEIGHTS_DIR, 'block_c_10classes.pth')
    NUM_CLASSES = 10
    DEVICE = "cuda"

    from config import cfg
    cfg.immutable(False)
    cfg.DATASET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.BNFUNC = torch.nn.BatchNorm2d
    cfg.MODEL.HRNET_CHECKPOINT = os.path.join(WEIGHTS_DIR, 'hrnetv2_w48_imagenet_pretrained.pth')
    cfg.OPTIONS.INIT_DECODER = False
    # val.py sets this via assert_and_infer_cfg(); required for network/mynn.py interpolate branches
    _m = re.match(r'^([0-9]+\.[0-9]+)', torch.__version__)
    cfg.OPTIONS.TORCH_VERSION = float(_m.group(1)) if _m else 2.0
    cfg.immutable(True)

    from network.ocrnet import HRNet_Mscale

    model = HRNet_Mscale(num_classes=NUM_CLASSES, criterion=None).to(DEVICE)

    # PyTorch 2.6+ defaults weights_only=True; CitySurfaces checkpoints need False (trusted local files).
    checkpoint = torch.load(WEIGHTS_FILE, map_location=DEVICE, weights_only=False)
    state_dict = checkpoint.get('state_dict', checkpoint)

    model_state = model.state_dict()
    new_state = {}
    for k in model_state:
        if k in state_dict and model_state[k].size() == state_dict[k].size():
            new_state[k] = state_dict[k]
        elif 'module.' + k in state_dict and model_state[k].size() == state_dict['module.' + k].size():
            new_state[k] = state_dict['module.' + k]

    model_state.update(new_state)
    model.load_state_dict(model_state)
    model.eval()

    return "Pretrained CitySurfaces model loaded (10 classes)"


_curio_output = _curio_node()

try:
    result_09c6e03f_117d_45c5_af30_8f71bc3e58b6 = _curio_output
except NameError:
    result_09c6e03f_117d_45c5_af30_8f71bc3e58b6 = None


In [ ]:
__trill_node__ = {
    "id": "9def5617-0b4e-4afb-afaf-7a567af01f92",
    "type": "DATA_LOADING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    import geopandas as gpd
    # Load neighborhood data
    boston = gpd.read_file('./data/dataset/Census2020_BlockGroups.shp').to_crs('EPSG:4326')
    return boston

_curio_output = _curio_node()

try:
    data_9def5617_0b4e_4afb_afaf_7a567af01f92 = _curio_output
except NameError:
    data_9def5617_0b4e_4afb_afaf_7a567af01f92 = None


In [ ]:
__trill_node__ = {
    "id": "c66ae5dc-5727-4dba-9ad7-e0d312cbc1cb",
    "type": "DATA_LOADING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    import pandas as pd
    df = pd.read_csv('./data/dataset/gsv/boston_gsv.csv', names=['status','id','lat','lon'])
    sample = df[df['status']=='OK'].sample(100, random_state=42)
    return sample

_curio_output = _curio_node()

try:
    data_c66ae5dc_5727_4dba_9ad7_e0d312cbc1cb = _curio_output
except NameError:
    data_c66ae5dc_5727_4dba_9ad7_e0d312cbc1cb = None


In [ ]:
__trill_node__ = {
    "id": "aaff4f52-b04b-413f-9f83-7e12fb0acbf0",
    "type": "COMPUTATION_ANALYSIS",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = data_c66ae5dc_5727_4dba_9ad7_e0d312cbc1cb
    arg = input_0

    import sys
    import os
    import re
    import torch
    import torch.nn.functional as F
    import numpy as np
    from PIL import Image
    from io import BytesIO
    import base64

    sample = arg

    _ROOT = os.getcwd()
    _CANDIDATES = [
        os.environ.get("CITYSURFACES_DIR"),
        os.path.join(_ROOT, "city-surfaces"),
        os.path.join(_ROOT, "CitySurfaces"),
    ]
    CITYSURFACES_DIR = None
    for _p in _CANDIDATES:
        if _p and os.path.isfile(os.path.join(_p, "config.py")):
            CITYSURFACES_DIR = os.path.abspath(_p)
            break
    if CITYSURFACES_DIR is None:
        raise FileNotFoundError(
            "CitySurfaces repo not found (need config.py). Clone "
            "https://github.com/VIDA-NYU/city-surfaces into ./city-surfaces/ under "
            f"your Curio launch directory, or set CITYSURFACES_DIR. cwd={_ROOT!r}"
        )
    sys.path.insert(0, CITYSURFACES_DIR)

    import tempfile
    from runx.logx import logx
    _log_dir = os.path.join(tempfile.gettempdir(), "curio_citysurfaces_runx")
    logx.initialize(logdir=_log_dir, tensorboard=False, hparams={}, global_rank=0)

    WEIGHTS_DIR = './data/dataset/CitySurfaces_weights'
    WEIGHTS_FILE = os.path.join(WEIGHTS_DIR, 'block_c_10classes.pth')
    NUM_CLASSES = 10
    DEVICE = 'cuda'
    IMAGE_SIZE = 320

    from config import cfg
    cfg.immutable(False)
    cfg.DATASET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.BNFUNC = torch.nn.BatchNorm2d
    cfg.MODEL.HRNET_CHECKPOINT = os.path.join(WEIGHTS_DIR, 'hrnetv2_w48_imagenet_pretrained.pth')
    cfg.OPTIONS.INIT_DECODER = False
    _m = re.match(r'^([0-9]+\.[0-9]+)', torch.__version__)
    cfg.OPTIONS.TORCH_VERSION = float(_m.group(1)) if _m else 2.0
    cfg.immutable(True)

    from network.ocrnet import HRNet_Mscale

    def compute_uncertainty(predictions):
       sorted_probs = np.sort(predictions, axis=1)
       highest_prob = sorted_probs[:, -1, :, :]
       second_highest_prob = sorted_probs[:, -2, :, :]
       uncertainty_margin = highest_prob - second_highest_prob
       return 1.0 - uncertainty_margin

    model = HRNet_Mscale(num_classes=NUM_CLASSES, criterion=None).to(DEVICE)

    checkpoint = torch.load(WEIGHTS_FILE, map_location=DEVICE, weights_only=False)
    state_dict = checkpoint.get('state_dict', checkpoint)
    model_state = model.state_dict()
    new_state = {}
    for k in model_state:
        if k in state_dict and model_state[k].size() == state_dict[k].size():
            new_state[k] = state_dict[k]
        elif 'module.' + k in state_dict and model_state[k].size() == state_dict['module.' + k].size():
            new_state[k] = state_dict['module.' + k]
    model_state.update(new_state)
    model.load_state_dict(model_state)
    model.eval()

    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
    STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)

    color_map = {
       0: (255, 127, 14),    # concrete
       1: (43, 160, 43),     # bricks
       2: (31, 119, 179),    # granite
       3: (153, 153, 153),   # asphalt
       4: (214, 39, 40),     # mixed
       5: (54, 54, 54),      # road
       6: (0, 0, 0),         # background
       7: (138, 0, 138),     # granite block-stone
       8: (240, 110, 170),   # hexagonal
       9: (139, 109, 48),    # cobblestone
    }

    lats = []
    lons = []
    uncerts = []
    images = []
    predicted_images = []
    uncert_images = []
    for index, row in sample.iterrows():
       image_path = './data/dataset/gsv/boston/%s_left.jpg' % row['id']

       pil_image = Image.open(image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))

       image = np.array(pil_image, dtype=np.float32) / 255.0
       input_tensor = torch.from_numpy(image.reshape(1, IMAGE_SIZE, IMAGE_SIZE, 3)).permute((0, 3, 1, 2)).to(DEVICE)
       input_tensor = (input_tensor - MEAN) / STD

       with torch.no_grad():
           output = model({'images': input_tensor})
           logits = output['pred']
           predictions = F.softmax(logits, dim=1)

       pred_labels = torch.argmax(predictions, dim=1)
       pred_array = pred_labels.cpu().numpy()
       pred_array = pred_array.reshape((IMAGE_SIZE, IMAGE_SIZE))
       pred_pil = Image.new("RGB", (pred_array.shape[1], pred_array.shape[0]))
       for i in range(pred_array.shape[0]):
           for j in range(pred_array.shape[1]):
               pred_pil.putpixel((j, i), color_map[pred_array[i, j]])

       buffered = BytesIO()
       pred_pil.save(buffered, format="PNG")
       pred_str = base64.b64encode(buffered.getvalue()).decode('utf-8')

       uncertainty_margin = compute_uncertainty(predictions.cpu().detach().numpy())

       uncertainty_array = np.uint8(uncertainty_margin * 255)
       uncertainty_array = np.transpose(uncertainty_array, (1, 2, 0))
       uncertainty_array = np.squeeze(uncertainty_array, axis=2)
       uncertainty_pil = Image.fromarray(uncertainty_array)

       buffered = BytesIO()
       uncertainty_pil.save(buffered, format="PNG")
       uncertainty_str = base64.b64encode(buffered.getvalue()).decode('utf-8')

       lats.append(row['lat'])
       lons.append(row['lon'])
       uncerts.append(float(np.average(uncertainty_margin)))

       buffered = BytesIO()
       pil_image.save(buffered, format="PNG")
       img_str = base64.b64encode(buffered.getvalue()).decode('utf-8')

       images.append(img_str)
       predicted_images.append(pred_str)
       uncert_images.append(uncertainty_str)

    return (lats, lons, uncerts, images, predicted_images, uncert_images)


_curio_output = _curio_node()

try:
    result_aaff4f52_b04b_413f_9f83_7e12fb0acbf0 = _curio_output
except NameError:
    result_aaff4f52_b04b_413f_9f83_7e12fb0acbf0 = None


In [ ]:
__trill_node__ = {
    "id": "cf7bf5ef-5ce7-4e26-974b-fb782f84be19",
    "type": "COMPUTATION_ANALYSIS",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = result_aaff4f52_b04b_413f_9f83_7e12fb0acbf0
    arg = input_0

    import geopandas as gpd

    lats = arg[0]
    lons = arg[1]
    uncerts = arg[2]
    original_images = arg[3]
    predicted_images = arg[4]
    uncert_images = arg[5]

    image_content = list(zip(original_images, predicted_images, uncert_images))

    gdf = pd.DataFrame({'lat': lats, 'lon': lons, 'uncertainty': uncerts, 'image_content': image_content})

    gdf['image_id'] = gdf.index

    gdf = gpd.GeoDataFrame(
       gdf, geometry=gpd.points_from_xy(gdf.lon, gdf.lat), crs="EPSG:4326"
    )

    gdf = gdf.sort_values(by='image_id', ascending=True)

    return gdf


_curio_output = _curio_node()

try:
    result_cf7bf5ef_5ce7_4e26_974b_fb782f84be19 = _curio_output
except NameError:
    result_cf7bf5ef_5ce7_4e26_974b_fb782f84be19 = None


In [ ]:
__trill_node__ = {
    "id": "7902bce6-4771-4f73-9ee1-d706fc22892f",
    "type": "MERGE_FLOW",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    inputs = [

    ]

    merged_inputs = [i for i in inputs if i is not None]

    return merged_inputs


_curio_output = _curio_node()

try:
    merged_7902bce6_4771_4f73_9ee1_d706fc22892f = _curio_output
except NameError:
    merged_7902bce6_4771_4f73_9ee1_d706fc22892f = None


In [ ]:
__trill_node__ = {
    "id": "55aa4581-9a68-4257-b2c7-63e3360737e3",
    "type": "COMPUTATION_ANALYSIS",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = merged_7902bce6_4771_4f73_9ee1_d706fc22892f
    arg = input_0

    import geopandas as gpd

    boston = arg[0]
    gdf = arg[1]

    def agg_to_list(series):
       return list(series)

    joined = gpd.sjoin(boston, gdf).groupby('GEOID20').agg({'uncertainty': 'mean', 'image_id': agg_to_list})
    boston = boston.set_index('GEOID20')
    boston.loc[joined.index,'uncertainty'] = joined['uncertainty']
    boston.loc[joined.index,'image_id'] = joined['image_id']

    filtered_boston = boston.loc[joined.index]

    filtered_boston = filtered_boston.rename(columns={'image_id': 'linked'})

    return filtered_boston


_curio_output = _curio_node()

try:
    result_55aa4581_9a68_4257_b2c7_63e3360737e3 = _curio_output
except NameError:
    result_55aa4581_9a68_4257_b2c7_63e3360737e3 = None


In [ ]:
__trill_node__ = {
    "id": "71ab6de4-b23c-42d9-a2fe-ce5141d285b2",
    "type": "DATA_CLEANING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = result_55aa4581_9a68_4257_b2c7_63e3360737e3
    arg = input_0

    import geopandas as gpd

    filtered_boston = arg

    filtered_boston = filtered_boston.loc[:, [filtered_boston.geometry.name, 'uncertainty', 'linked']]

    filtered_boston = filtered_boston.set_crs(4326)
    filtered_boston = filtered_boston.to_crs(3395)

    filtered_boston.metadata = {
       'name': 'boston'
    }

    return filtered_boston


_curio_output = _curio_node()

try:
    result_71ab6de4_b23c_42d9_a2fe_ce5141d285b2 = _curio_output
except NameError:
    result_71ab6de4_b23c_42d9_a2fe_ce5141d285b2 = None


In [ ]:
__trill_node__ = {
    "id": "ddae8fb9-82ee-4523-bf07-9184c7fc873f",
    "type": "VIS_UTK",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():



    import utk

    {
        "components": [
            {
                "id": "grammar_map",
                "position": {
                    "width": [
                        1,
                        12
                    ],
                    "height": [
                        1,
                        4
                    ]
                }
            }
        ],
        "knots": [],
        "ex_knots": [
            {
                "id": "boston0",
                "out_name": "boston",
                "in_name": "uncertainty"
            }
        ],
        "grid": {
            "width": 12,
            "height": 4
        },
        "grammar": false
    }


_curio_output = _curio_node()

try:
    result_ddae8fb9_82ee_4523_bf07_9184c7fc873f = _curio_output
except NameError:
    result_ddae8fb9_82ee_4523_bf07_9184c7fc873f = None


In [ ]:
__trill_node__ = {
    "id": "3c5db6ce-0082-47d7-80b7-1b9534b4726b",
    "type": "DATA_POOL",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    return None


_curio_output = _curio_node()

try:
    pool_3c5db6ce_0082_47d7_80b7_1b9534b4726b = _curio_output
except NameError:
    pool_3c5db6ce_0082_47d7_80b7_1b9534b4726b = None


In [ ]:
__trill_node__ = {
    "id": "f7172aea-3e3d-4c58-b0bc-dc02db24a733",
    "type": "DATA_POOL",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    return None


_curio_output = _curio_node()

try:
    pool_f7172aea_3e3d_4c58_b0bc_dc02db24a733 = _curio_output
except NameError:
    pool_f7172aea_3e3d_4c58_b0bc_dc02db24a733 = None


In [ ]:
__trill_node__ = {
    "id": "a61f234d-78b4-4c3f-9d22-15a2855967b3",
    "type": "COMPUTATION_ANALYSIS",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = pool_f7172aea_3e3d_4c58_b0bc_dc02db24a733
    arg = input_0

    df = pd.DataFrame(arg.drop(columns=arg.geometry.name))
    df = df[df['interacted'] == '1']
    df = df.sort_values(by='uncertainty', ascending=False)
    return df.head(20)


_curio_output = _curio_node()

try:
    result_a61f234d_78b4_4c3f_9d22_15a2855967b3 = _curio_output
except NameError:
    result_a61f234d_78b4_4c3f_9d22_15a2855967b3 = None


In [ ]:
__trill_node__ = {
    "id": "e2e0b5d8-a0dc-4860-9a08-203871b0d28f",
    "type": "VIS_IMAGE",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = result_a61f234d_78b4_4c3f_9d22_15a2855967b3
    arg = input_0




_curio_output = _curio_node()

try:
    result_e2e0b5d8_a0dc_4860_9a08_203871b0d28f = _curio_output
except NameError:
    result_e2e0b5d8_a0dc_4860_9a08_203871b0d28f = None
